# Cryptography-02 — Du canal bruité au canal adversarial

**Série Cryptography** · Carnet 02 — Du canal bruité au canal adversarial

Jusqu'ici, le canal était **bruité** : un bruit aléatoire retournait des bits, et la
théorie des codes (codes correcteurs, distance de Hamming) ajoutait de la
redondance pour **corriger** ces erreurs. Ce carnet change de **modèle de menace** :
le canal est désormais **adversarial** — un **tiers** (un espion) observe tout ce
qui transite, et peut aussi **altérer** le message. La redondance ne suffit plus :
un espion qui voit toute la transmission « corrige » les erreurs avec nous. Ce qu'il
faut, c'est un **secret** que l'adversaire ne possède pas. Nous construisons, en C#
sur la bibliothèque standard .NET (`System.Security.Cryptography`), les quatre
propriétés de sécurité — **confidentialité**, **intégrité**, **authenticité**,
**non-répudiation** — et nous voyons la primitive .NET qui porte chacune.


## Enjeu — du bruit corrigé à l'adversaire écarté

Le passage du canal **bruité** au canal **adversarial** change la nature du problème :

- **Canal bruité** : l'ennemi est le **bruit** (aléatoire, non malveillant). La
  parade est la **redondance** — des bits en plus qui permettent de détecter puis
  de corriger les retournements. *Renvoi → InformationTheory (#27077) : les codes
  correcteurs.*
- **Canal adversarial** : l'ennemi est un **adversaire** (malveillant, qui **lit**
  et **manipule**). La redondance ne sert à rien — l'adversaire voit tout et
  « aide » le décodeur. La parade est la **sécurité** : un **secret partagé** (une
  clé) que l'adversaire ne possède pas.

Tout le carnet repose sur ce basculement : **corriger le bruit** (redondance) vs
**écarter l'adversaire** (sécurité). Et les quatre propriétés de sécurité sont les
quatre questions que ce secret doit répondre.


## Les quatre propriétés de sécurité

Face à un adversaire, la sécurité d'un message se décompose en quatre propriétés,
chacune répondant à une menace distincte :

| Propriété | Menace qu'elle écarte | Question |
|---|---|---|
| **Confidentialité** | l'espion **lit** le message | « L'adversaire peut-il **comprendre** le message ? » |
| **Intégrité** | l'adversaire **modifie** le message en route | « Le message a-t-il été **altéré** ? » |
| **Authenticité** | l'adversaire **se fait passer** pour l'expéditeur | « Le message vient-il **vraiment** de l'expéditeur qu'il dit ? » |
| **Non-répudiation** | l'expéditeur **nie** avoir envoyé | « L'expéditeur peut-il **nier** l'avoir envoyé ? » |

Nous obtenons chacune avec une primitive .NET précise — et c'est la **4ᵉ**
(non-répudiation) qui force à passer du **secret partagé** (symétrique) à la
**clé privée** (asymétrique).


In [1]:
using System;
using System.Text;
using System.Security.Cryptography;

// Helpers d'affichage en hex (Convert.ToHexString est indisponible sur .NET < 5).
static class Hex {
    public static string To(byte[] b) {
        var sb = new StringBuilder();
        foreach (var x in b) sb.Append(x.ToString("x2"));
        return sb.ToString();
    }
    public static byte[] From(string s) {
        byte[] b = new byte[s.Length / 2];
        for (int i = 0; i < b.Length; i++)
            b[i] = Convert.ToByte(s.Substring(i * 2, 2), 16);
        return b;
    }
}

// Le message que nous protègerons tout le long du carnet.
string message = "Le canal est adversarial : un tiers observe la transmission.";
byte[] plain = Encoding.UTF8.GetBytes(message);
Console.WriteLine("Message clair (" + plain.Length + " octets) : " + message);
Console.WriteLine("En hex       : " + Hex.To(plain));


The below script needs to be able to find the current output cell; this is an easy method to get it.

Message clair (60 octets) : Le canal est adversarial : un tiers observe la transmission.


En hex       : 4c652063616e616c2065737420616476657273617269616c203a20756e207469657273206f627365727665206c61207472616e736d697373696f6e2e


## 1. Confidentialité — AES : l'espion ne lit pas

La **confidentialité** veut que l'adversaire, qui **voit** la transmission, ne
puisse **pas comprendre** le message. On y parvient avec un chiffrement
**symétrique** : l'expéditeur et le destinataire partagent une **clé secrète**
commune. L'algorithme **AES** (Advanced Encryption Standard) est le chiffrement
symétrique de référence. Sans la clé, le texte chiffré est, pour l'espion, du
bruit ; avec la clé, le destinataire le déchiffre exactement.


In [2]:
// --- 1. CONFIDENTIALITE : AES (symetrique) ---
// Sans la cle, le message est illisible ; avec, le destinataire le recupere.
byte[] aesKey = new byte[32];   // cle AES-256
byte[] aesIv  = new byte[16];   // vecteur d'initialisation

byte[] ct;
using (var aes = Aes.Create()) {
    aes.Key = aesKey;
    aes.IV  = aesIv;
    using (var enc = aes.CreateEncryptor())
        ct = enc.TransformFinalBlock(plain, 0, plain.Length);
}
string recovered;
using (var aes = Aes.Create()) {
    aes.Key = aesKey;
    aes.IV  = aesIv;
    using (var dec = aes.CreateDecryptor()) {
        byte[] pt = dec.TransformFinalBlock(ct, 0, ct.Length);
        recovered = Encoding.UTF8.GetString(pt);
    }
}
Console.WriteLine("== 1. CONFIDENTIALITE (AES-256) ==");
Console.WriteLine("Clair   : " + message);
Console.WriteLine("Chiffré : " + Hex.To(ct));
Console.WriteLine("Dechiffré par le destinataire (qui a la cle) : " + recovered);
if (recovered != message) throw new Exception("le round-trip AES a altere le message");
Console.WriteLine("-> l'espion voit le chiffré, pas le clair : confidentialite OBTENUE.");


== 1. CONFIDENTIALITE (AES-256) ==


Clair   : Le canal est adversarial : un tiers observe la transmission.


Chiffré : b578a5fb0e6fe716483e011ed55849a88660c35900af8fdbf325b6d66e48fbb3a6cf17e560ec61118118ee6afb5e66dd2d4ddb78205cddcf8d8e4eb0ed6b4e53


Dechiffré par le destinataire (qui a la cle) : Le canal est adversarial : un tiers observe la transmission.


-> l'espion voit le chiffré, pas le clair : confidentialite OBTENUE.


## 2. Intégrité + Authenticité — HMAC : toute altération casse le MAC

L'**intégrité** veut détecter que le message n'a pas été **modifié** en route ;
l'**authenticité** veut garantir qu'il vient **vraiment** de l'expéditeur dit. Les
deux se obtains ensemble avec un **MAC** (Message Authentication Code) : une
empreinte du message calculée **avec une clé secrète partagée**. La primitive est
**HMAC-SHA256**. À la réception, on recalcule l'empreinte avec la clé : si elle ne
coller pas, le message a été altéré (intégrité) — et comme l'adversaire n'a pas la
clé, il ne peut **pas** produire un MAC valide sur un faux message (authenticité).


In [3]:
// --- 2. INTEGRITE + AUTHENTICITE : HMAC-SHA256 (cle partagee) ---
// Un MAC calcule avec une cle partagee : toute altération change le MAC.
byte[] macKey = new byte[32];
byte[] mac;
using (var hmac = new HMACSHA256(macKey))
    mac = hmac.ComputeHash(plain);

byte[] tampered = (byte[])plain.Clone();
tampered[0] = (byte)(tampered[0] ^ 1);   // l'adversaire retourne un bit en route

bool intact, altered;
using (var hmac2 = new HMACSHA256(macKey)) {
    // Verifier un HMAC = recalculer le MAC et le comparer (en temps fixe).
    intact  = CryptographicOperations.FixedTimeEquals(mac, hmac2.ComputeHash(plain));
    altered = CryptographicOperations.FixedTimeEquals(mac, hmac2.ComputeHash(tampered));
}
Console.WriteLine("== 2. INTEGRITE + AUTHENTICITE (HMAC-SHA256) ==");
Console.WriteLine("MAC (hex) : " + Hex.To(mac));
Console.WriteLine("Message intact  -> MAC valide ? " + intact);
Console.WriteLine("Message altéré  -> MAC valide ? " + altered);
if (!intact || altered) throw new Exception("HMAC : le MAC ne discrimine pas");
Console.WriteLine("-> un bit retourné CASSE le MAC : intégrité + authenticite OBTENUES.");
Console.WriteLine("   (l'adversaire sans la cle ne peut PAS forger un MAC valide)");


== 2. INTEGRITE + AUTHENTICITE (HMAC-SHA256) ==


MAC (hex) : 6cbcb87d2af6c46b77fb8e5c5d5374e2a7c6badc37d870bdbaee64aac817ab4e


Message intact  -> MAC valide ? True


Message altéré  -> MAC valide ? False


-> un bit retourné CASSE le MAC : intégrité + authenticite OBTENUES.


   (l'adversaire sans la cle ne peut PAS forger un MAC valide)


## 3. Non-répudiation — signature RSA : l'expéditeur ne peut ni forger ni nier

L'HMAC a une limite : il repose sur une **clé partagée** — l'expéditeur ET le
destinataire la possèdent, donc **les deux** peuvent produire le MAC. L'expéditeur
peut donc **nier** avoir envoyé (« c'est le destinataire qui a fait ce MAC »). La
**non-répudiation** résout ça avec l'**asymétrie** : l'expéditeur signe avec sa
**clé privée** (seule à lui), et **n'importe qui** — un tiers, un juge — vérifie
avec sa **clé publique**. Seul le porteur de la clé privée a pu produire la
signature ; l'expéditeur ne peut ni la **forger** (sans sa clé privée) ni la
**nier** (un tiers a vérifié). La primitive est la **signature RSA** (couplée à
SHA-256).


In [4]:
// --- 3. NON-REPUDIATION : signature RSA (asymetrique) ---
// L'expéditeur signe avec sa cle PRIVEE ; un TIERS vérifie avec la cle PUBLIQUE.
using (var rsa = RSA.Create()) {          // paire RSA-2048
    byte[] sig = rsa.SignData(plain, HashAlgorithmName.SHA256, RSASignaturePadding.Pkcs1);

    // Un tiers ne possede QUE la cle publique (ExportParameters(false)).
    RSA pubOnly = RSA.Create(rsa.ExportParameters(false));

    byte[] tampered2 = (byte[])plain.Clone();
    tampered2[3] = (byte)(tampered2[3] ^ 1);
    bool byThirdParty, byThirdPartyTampered;
    byThirdParty         = pubOnly.VerifyData(plain, sig, HashAlgorithmName.SHA256, RSASignaturePadding.Pkcs1);
    byThirdPartyTampered = pubOnly.VerifyData(tampered2, sig, HashAlgorithmName.SHA256, RSASignaturePadding.Pkcs1);

    Console.WriteLine("== 3. NON-REPUDIATION (signature RSA-2048) ==");
    Console.WriteLine("Signature (hex, tronquée) : " + Hex.To(sig).Substring(0, 32) + "…");
    Console.WriteLine("Un TIERS (cle publique seule) vérifie le message   ? " + byThirdParty);
    Console.WriteLine("Le même tiers vérifie un message ALTÉRÉ            ? " + byThirdPartyTampered);
    if (!byThirdParty || byThirdPartyTampered) throw new Exception("RSA : la signature ne discrimine pas");
    Console.WriteLine("-> seul le porteur de la cle PRIVEE a pu signer : l'expéditeur");
    Console.WriteLine("   ne peut NI forger NI nier : non-répudiation OBTENUE.");
}


== 3. NON-REPUDIATION (signature RSA-2048) ==


Signature (hex, tronquée) : 21cc7f5d44674dda4fe8c6e126b3f7e6…


Un TIERS (cle publique seule) vérifie le message   ? True


Le même tiers vérifie un message ALTÉRÉ            ? False


-> seul le porteur de la cle PRIVEE a pu signer : l'expéditeur


   ne peut NI forger NI nier : non-répudiation OBTENUE.


## 4. Le contraste — pourquoi la sécurité, pas la redondance

C'est le point qui relie ce carnet au précédent : face à un **adversaire**, la
**redondance** (codes correcteurs) est **inutile** — l'espion voit la transmission
complète, il n'y a rien à « corriger », il **lit tout**. Ce qui écarte
l'adversaire, c'est le **secret** (la clé) qu'il ne possède pas.


In [5]:
// --- 4. LE CONTRASTE : la redondance n'ecarte PAS l'adversaire ---
Console.WriteLine("== 4. POURQUOI LA SEGRETE, PAS LA REDONDANCE ==");
Console.WriteLine("Canal BRUITÉ      : l'ennemi est le bruit -> on ajoute de la REDONDANCE");
Console.WriteLine("                   (codes correcteurs, theorie des codes).");
Console.WriteLine("Canal ADVERSARIAL : l'ennemi EST l'espion -> il voit TOUT :");
Console.WriteLine("                   la redondance l'AJUTE a decoder ; seule la CLE");
Console.WriteLine("                   (secret partage ou cle privee) l'ecarte.");
Console.WriteLine();
Console.WriteLine("AES ci-dessus : l'espion a le chiffré COMPLET, il ne lit RIEN");
Console.WriteLine("  (sans la cle). C'est le secret qui protege, pas la redondance.");


== 4. POURQUOI LA SEGRETE, PAS LA REDONDANCE ==


Canal BRUITÉ      : l'ennemi est le bruit -> on ajoute de la REDONDANCE


                   (codes correcteurs, theorie des codes).


Canal ADVERSARIAL : l'ennemi EST l'espion -> il voit TOUT :


                   la redondance l'AJUTE a decoder ; seule la CLE


                   (secret partage ou cle privee) l'ecarte.


AES ci-dessus : l'espion a le chiffré COMPLET, il ne lit RIEN


  (sans la cle). C'est le secret qui protege, pas la redondance.


## Synthèse — une primitive .NET par propriété

| Propriété | Menace | Primitive .NET | Clé |
|---|---|---|---|
| **Confidentialité** | l'espion lit | **AES** (`Aes`) | **partagée** (symétrique) |
| **Intégrité** | l'adversaire modifie | **HMAC-SHA256** (`HMACSHA256`) | **partagée** |
| **Authenticité** | usurpation d'identité | **HMAC** (MAC) | **partagée** |
| **Non-répudiation** | l'expéditeur nie | **signature RSA** (`RSA`) | **privée/publique** (asymétrique) |

Le fil : les trois premières propriétés suffisent avec un **secret partagé** ; la
**non-répudiation** force à la **clé privée** — car seul son porteur peut signer,
et n'importe qui peut vérifier avec la clé publique. C'est le basculement vers
l'**asymétrie** que le carnet 01 avait présenté (RSA, ECC) et que ce carnet met au
service de la confiance.


## Renvois croisés (non-doublon)

- **↔ InformationTheory (#27077)** : le **canal bruité** et les **codes correcteurs**
  (distance de Hamming, code de Hamming) sont le territoire de la théorie de
  l'information — ce carnet **renvoie** pour la partie « corriger le bruit », et
  reprend le **code-based / McEliece** (crypto à base de codes) au carnet 04.
- **↔ SmartContracts (`04-Privacy-Cryptography`)** : les **preuves à divulgation
  nulle** (ZK) et la confidentialité sur chaîne vivent dans le **pilier commun**
  `SmartContracts` — renvois dans les deux sens (carnet 03 de cette série).
- **↔ Complexity** : la **dureté** (décodage, problème du log discret,
  factorisation) sur laquelle repose la sécurité — le post-quantique (carnet 04)
  renvoie à `Complexity`.
- **↔ lake Lean** : les **preuves formelles** de propriétés crypto dans le lake
  Lean du dépôt.


## Exercices


In [6]:
// Exercice 1 — Pourquoi la clé doit rester secrète
// AES et HMAC reposent sur une CLE PARTAGEE. Argumenter : si l'adversaire
// decouvre la cle (ex. fuite), QUELLE propriété tombe en premier, et pourquoi
// la non-répudiation (signature RSA) survit a la fuite de la cle PUBLIQUE
// (mais pas a celle de la cle PRIVEE) ?
// TODO etudiant


In [7]:
// Exercice 2 — Forger un MAC
// Montrer concretement que, SANS la cle HMAC, l'adversaire ne peut PAS produire
// un (message, MAC) valide : prendre un message altéré, calculer son HMAC avec
// une cle FABLEE, et verifier que VerifyData renvoie false (le MAC ne colle pas).
// TODO etudiant


In [8]:
// Exercice 3 — Confidentialité vs intégrité
// Chiffrer (AES) ALORS que le message est déjà altéré : l'altération survit-elle
// au chiffrement ? (Oui : AES est une permutation — un bit change en entree le
// reste changeant en sortie.) Ajouter un HMAC APRES chiffrement et discuter
// laquelle des deux ordres (MAC-then-encrypt vs encrypt-then-MAC) donne a la
// fois confidentialite ET integrite.
// TODO etudiant


## À retenir

- Le **canal adversarial** change le problème : l'ennemi n'est plus le bruit
  (à **corriger** par redondance) mais un **adversaire** (à **écarter** par
  secret).
- **Confidentialité** = **AES** : sans la clé, le chiffré est illisible.
- **Intégrité + authenticité** = **HMAC** (clé partagée) : toute altération casse
  le MAC, et l'adversaire ne peut pas en forger un valide.
- **Non-répudiation** = **signature RSA** (clé privée/publique) : seul le porteur
  de la clé privée signe ; un tiers vérifie avec la clé publique — l'expéditeur
  ne peut ni forger ni nier.
- Le passage du **secret partagé** (symétrique) à la **clé privée**
  (asymétrique) est ce qui rend la non-répudiation possible.


## Bibliographie

- NIST, *SP 800-57 : Recommendation for Key Management* — les propriétés de
  sécurité et la gestion des clés (symétrique vs asymétrique).
- B. Schneier, *Applied Cryptography*, 2ᵉ éd., John Wiley & Sons, 1996 — le
  référentiel des primitives (AES, HMAC, RSA) et de leur composition.
- Carnet 03 (à venir) : le chiffrement homomorphe et les preuves à divulgation
  nulle — calculer sur des données chiffrées sans les déchiffrer.
